# Hybrid SQG path: burn-in + solver vs DM comparison

1. **Burn-in** — true SQG solver until spin-up.
2. **Fork** — same post-burn-in state for two rollouts:
   - continue with the **true SQG solver**;
   - continue with **conditional diffusion** (autoregressive).
3. **Compare** — RMSE / snapshots / side-by-side animation.

Set `NETWORK_PKL`, `NX`, `HISTORY` to match training (`sqg_path_next`).


In [ ]:
# --- User configuration ---
import os

from project_utils.paths import PROJECT_ROOT, dataset_trajectory_dir, CKPT_DIR

SQG_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/sqg_default.yaml")
DATA_ROOT = "SQG_64d_1h"
TRAJECTORY_SUBDIR = "full_path"

NETWORK_PKL = os.path.join(
    CKPT_DIR,
    "sqg_dim_64_dt_1.pkl",
)

DEVICE = "cuda:0"
SEED = 0

NX = 64
T_BURN_IN = 400
T_ROLLOUT = 200  # comparison steps after burn-in (solver vs DM)
OUTPUT_INTERVAL_HOURS = 1.0

HISTORY = 1
VAR_SELECT = [0, 1]

NUM_STEPS = 18
SEED_DM = 0



In [ ]:
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.yaml_simple import load_simple_yaml
from turbulence.sqg import SQGParams, SQGSolver

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)
print("Trajectory dir:", trajectory_dir)
print("norm_stats:", norm_stats_path)

In [ ]:
def edm_sampler(
    net,
    latents,
    conditions,
    class_labels=None,
    randn_like=torch.randn_like,
    num_steps=18,
    sigma_min=0.002,
    sigma_max=80,
    rho=7,
    S_churn=0,
    S_min=0,
    S_max=float("inf"),
    S_noise=1,
):
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    x_next = latents.to(torch.float64) * t_steps[0]
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        if i < num_steps - 1:
            denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)
    return x_next


class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(input.shape, dtype=input.dtype, layout=input.layout, device=input.device)


def make_initial_pv(N: int, device: torch.device, seed: int) -> torch.Tensor:
    torch.manual_seed(seed)
    pv = torch.randn(2, N, N, dtype=torch.float32, device=device) * 100.0
    nexp = 20
    x = torch.arange(0.0, 2.0 * np.pi, 2.0 * np.pi / N, device=device, dtype=torch.float32)
    y = torch.arange(0.0, 2.0 * np.pi, 2.0 * np.pi / N, device=device, dtype=torch.float32)
    x, y = torch.meshgrid(x, y, indexing="xy")
    pv[1] = pv[1] + 2000.0 * (torch.sin(x / 2.0) ** (2 * nexp) * torch.sin(y) ** nexp)
    return pv - pv.mean(dim=(1, 2), keepdim=True)


def build_condition(frames_phys, history: int, stats: dict) -> torch.Tensor:
    """Match DatasetPathCond: stack last `history` normalized frames -> (1, history*C, H, W)."""
    recent = frames_phys[-history:]
    normed = [normalize_tensor(f, stats) for f in recent]
    block = torch.stack(normed, dim=0)  # (history, C, H, W)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    cond = block.reshape(history * C, H, W)
    return cond.unsqueeze(0)  # (1, history*C, H, W)

## Phase 1: SQG solver burn-in

Integrate the true model for `T_BURN_IN` saved frames (same spacing as `generate_fields.py`), then keep the state for diffusion.

In [ ]:
cfg = load_simple_yaml(SQG_CONFIG)
sqg_cfg = dict(cfg.get("sqg", {}))
sqg_cfg["device"] = str(device)

params = SQGParams(**sqg_cfg)
params.validate(N=NX)
outputinterval = OUTPUT_INTERVAL_HOURS * 3600.0
timesteps_per_output = max(1, int(outputinterval / params.dt))

pv0 = make_initial_pv(NX, device, SEED)
solver = SQGSolver(pv0, **params.to_core_kwargs())

path_burn = [solver.fields()["pv"].detach().cpu()]
for _ in tqdm(range(T_BURN_IN), desc="SQG burn-in"):
    solver.step(timesteps_per_output)
    frame = solver.fields()["pv"].detach().cpu()
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite PV during burn-in")
    path_burn.append(frame)

# Shared state after burn-in (fork for solver continuation vs DM)
state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(path_burn, dim=0)  # (T_burn+1, 2, H, W)
t_burn_end = path_burn.shape[0] - 1

print(f"Burn-in: {path_burn.shape[0]} frames, last min/max: {path_burn[-1].min():.4g} / {path_burn[-1].max():.4g}")


## Phase 2: Load trained conditional diffusion model

In [ ]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL to your training snapshot. Not found: {NETWORK_PKL}"
    )

norm_stats = load_norm_stats(norm_stats_path)
print(f"Loaded norm_stats: vmin={norm_stats['vmin']:.6g}, vmax={norm_stats['vmax']:.6g}")

print(f"Loading network from {NETWORK_PKL} ...")
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    ckpt = pickle.load(f)
net = ckpt["ema"].to(device).eval()

n_target = len(VAR_SELECT)
n_cond = HISTORY * len(VAR_SELECT)
print(f"Model: {net.model_name}, target_ch={net.img_input_channels}, cond_ch={net.img_cond_channels}")
assert net.img_input_channels == n_target, "VAR_SELECT / checkpoint mismatch"
assert net.img_cond_channels == n_cond, "HISTORY / checkpoint mismatch"
assert net.img_res_h == NX and net.img_res_w == NX, "NX must match trained resolution"

## Phase 3: Parallel rollout after burn-in

From the **same** post-burn-in state:
- **True solver** — continue integrating SQG for `T_ROLLOUT` saved frames.
- **DM** — autoregressive conditional diffusion for `T_ROLLOUT` frames (condition = last `HISTORY` frames, updated each step).


In [ ]:
if path_burn.shape[0] < HISTORY:
    raise ValueError(f"Need at least HISTORY={HISTORY} burn-in frames")

H, W = path_burn.shape[-2], path_burn.shape[-1]
core_kw = params.to_core_kwargs()

# --- True solver branch (restore forked state) ---
solver_cont = SQGSolver(path_burn[-1].to(device), **core_kw)
solver_cont.set_state(state_after_burn)

rollout_solver = []
for _ in tqdm(range(T_ROLLOUT), desc="True solver rollout"):
    solver_cont.step(timesteps_per_output)
    frame = solver_cont.fields()["pv"].detach().cpu()
    if not torch.isfinite(frame).all():
        raise RuntimeError("Non-finite PV during solver rollout")
    rollout_solver.append(frame)
path_solver = torch.stack(rollout_solver, dim=0)  # (T_ROLLOUT, 2, H, W)

# --- DM branch (same initial history as end of burn-in) ---
path_dm_hist = [path_burn[i].clone() for i in range(path_burn.shape[0])]
rnd = StackedRandomGenerator(device, [SEED_DM])

rollout_dm = []
t0 = time.time()
for step in tqdm(range(T_ROLLOUT), desc="DM rollout"):
    cond = build_condition(path_dm_hist, HISTORY, norm_stats).to(device)
    latents = rnd.randn([1, n_target, H, W], device=device)
    with torch.no_grad():
        sample_norm = edm_sampler(net, latents, cond, num_steps=NUM_STEPS).to(torch.float32)
    next_phys = denormalize_tensor(sample_norm[0].cpu(), norm_stats)
    if not torch.isfinite(next_phys).all():
        raise RuntimeError(f"Non-finite DM sample at step {step}")
    rollout_dm.append(next_phys)
    path_dm_hist.append(next_phys)

path_dm = torch.stack(rollout_dm, dim=0)  # (T_ROLLOUT, 2, H, W)
print(f"DM rollout done in {time.time() - t0:.1f}s")
print("Shapes:", path_solver.shape, path_dm.shape)


## Phase 4: Compare true solver vs DM

Aligned frame `k` = `k` steps after the shared burn-in end (index `t_burn_end`).


In [ ]:
diff = path_solver - path_dm
rmse = (diff.pow(2).mean(dim=(2, 3)).sqrt())  # (T_ROLLOUT, 2)
mae = diff.abs().mean(dim=(2, 3))

rel_rmse = rmse / (path_solver.pow(2).mean(dim=(2, 3)).sqrt().clamp_min(1e-8))

steps = np.arange(T_ROLLOUT)
fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for c in range(2):
    axs[0].plot(steps, rmse[:, c].numpy(), label=f"channel {c}")
    axs[1].plot(steps, rel_rmse[:, c].numpy(), label=f"channel {c}")
axs[0].set_ylabel("RMSE (physical PV)")
axs[0].set_title("Solver vs DM — error after shared burn-in")
axs[0].legend()
axs[0].grid(True)
axs[1].set_ylabel("relative RMSE")
axs[1].set_xlabel("rollout step k (after burn-in)")
axs[1].legend()
axs[1].grid(True)
plt.tight_layout()
plt.show()

k_show = min(10, T_ROLLOUT - 1)
vmin = float(torch.min(path_solver[k_show]))
vmax = float(torch.max(path_solver[k_show]))
fig, axs = plt.subplots(2, 2, figsize=(10, 8))
for c in range(2):
    axs[0, c].imshow(path_solver[k_show, c].numpy(), cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axs[0, c].set_title(f"true solver, ch={c}, k={k_show}")
    axs[0, c].axis("off")
    axs[1, c].imshow(path_dm[k_show, c].numpy(), cmap="RdBu_r", vmin=vmin, vmax=vmax)
    axs[1, c].set_title(f"DM, ch={c}, k={k_show}")
    axs[1, c].axis("off")
fig.suptitle("Snapshot comparison (same post-burn-in step)")
plt.tight_layout()
plt.show()


In [ ]:
# SAVE_PATH = os.path.join(
#     '.',
#     "hybrid_solver_dm_compare.pt",
# )

# meta = {
#     "T_burn_in": T_BURN_IN,
#     "T_rollout": T_ROLLOUT,
#     "history": HISTORY,
#     "network_pkl": NETWORK_PKL,
#     "sqg_config": SQG_CONFIG,
#     "seed_solver": SEED,
#     "seed_dm": SEED_DM,
# }
# torch.save(
#     {
#         "path_burn": path_burn,
#         "path_solver": path_solver,
#         "path_dm": path_dm,
#         "rmse": rmse,
#         "meta": meta,
#     },
#     SAVE_PATH,
# )
# print("Saved:", SAVE_PATH)


## Animate comparison (post burn-in)

Top row: **true SQG solver**. Bottom row: **conditional diffusion**. Title shows rollout step `k` after burn-in.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

interval_ms = 200
max_frames = min(T_ROLLOUT, 200)
frame_ids = list(range(max_frames))

stacked = torch.cat([path_solver[:max_frames], path_dm[:max_frames]], dim=0)
vmin = float(stacked.min())
vmax = float(stacked.max())

fig, axs = plt.subplots(2, 2, figsize=(9, 7))
ims = []
for row, label in enumerate(["true SQG solver", "conditional diffusion (DM)"]):
    data = path_solver if row == 0 else path_dm
    for c in range(2):
        ax = axs[row, c]
        im = ax.imshow(
            data[0, c].numpy(),
            cmap="RdBu_r",
            vmin=vmin,
            vmax=vmax,
            animated=True,
        )
        ims.append(im)
        ax.set_title(f"{label} — channel {c}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

title = fig.suptitle("")


def update(frame_idx):
    k = frame_ids[frame_idx]
    for c in range(2):
        ims[c].set_array(path_solver[k, c].numpy())
        ims[2 + c].set_array(path_dm[k, c].numpy())
    title.set_text(
        f"rollout step k = {k} (after burn-in)  |  top: true solver  |  bottom: DM"
    )
    return [*ims, title]


anim = FuncAnimation(
    fig,
    update,
    frames=len(frame_ids),
    interval=interval_ms,
    blit=False,
    repeat=True,
)
plt.close(fig)
HTML(anim.to_jshtml())
